# QLoRA training on a Kaggle T4

Trains a LoRA adapter on Qwen3-1.7B with `train/train_qlora.py` (Unsloth backend, 4-bit base,
fp16) and pushes it to the Hugging Face Hub. Hyperparameters come only from `train/config.toml`
at the pinned commit.

Before running:

- Settings: accelerator **GPU T4**, **Internet on**.
- Add-ons, Secrets: `HF_TOKEN` (a write token) and `SCRUB_TERMS` (the scrub terms file, one term
  per line). The push is refused if any pushed file contains a term. Neither secret is printed.
- Set the parameters below: `REPO_URL` must be a clone URL this notebook can reach, and
  `REPO_COMMIT` the full SHA of the commit to train from.

In [ ]:
REPO_URL = "https://github.com/OWNER/small_model_finetune.git"
REPO_COMMIT = "dd36d336ea84e293b99727068507c420670111c6"
DATASET = "data/datasets/v1/train.jsonl"
SEED = 0
HUB_REPO = "OWNER/groundcite-qwen3-1.7b-v1"
OUT = "/kaggle/working/run"
# Kaggle-only training stack. Unsloth 2026.10.10 supports transformers <= 5.17.0, so training
# pins that; eval keeps uv.lock's version. train_log.json records the versions used.
UNSLOTH_VERSION = "2026.10.10"
TRANSFORMERS_VERSION = "5.17.0"

Secrets go into the environment of the training process only; nothing is printed.

In [ ]:
import os
import tempfile

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
# Outside /kaggle/working, so it is never saved as notebook output.
with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as terms:
    terms.write(secrets.get_secret("SCRUB_TERMS"))
os.environ["SCRUB_TERMS_PATH"] = terms.name

Clone the repo at the pinned commit and install the locked dependencies plus Unsloth.

In [ ]:
import subprocess

REPO_DIR = "/tmp/repo"


def sh(cmd: str) -> None:
    subprocess.run(cmd, shell=True, check=True)


sh(f"git clone --quiet {REPO_URL} {REPO_DIR}")
sh(f"git -C {REPO_DIR} checkout --quiet {REPO_COMMIT}")
sh("pip install --quiet uv")
# uv.lock without torch: Kaggle's CUDA build of torch stays.
uv_export = "uv export --frozen --no-dev --no-hashes --no-emit-project --no-emit-package torch"
sh(f"cd {REPO_DIR} && {uv_export} -o /tmp/requirements.txt")
sh("pip install --quiet -r /tmp/requirements.txt")
sh(f"pip install --quiet --no-deps -e {REPO_DIR}")
unsloth = f"unsloth=={UNSLOTH_VERSION} unsloth_zoo=={UNSLOTH_VERSION}"
sh(f"pip install --quiet {unsloth} transformers=={TRANSFORMERS_VERSION}")

Train, then push the adapter, model card and `train_log.json` in one Hub commit.

In [ ]:
train = f"python train/train_qlora.py --dataset {DATASET} --out {OUT} --seed {SEED}"
sh(f"cd {REPO_DIR} && {train} --hub-repo {HUB_REPO}")

In [ ]:
import json

with open(f"{OUT}/train_log.json", encoding="utf-8") as f:
    log = json.load(f)
hub = log["hub"] or {}
final_loss = log["loss_curve"][-1]["loss"] if log["loss_curve"] else float("nan")
print(
    f"{hub.get('repo')}@{hub.get('commit_sha')} | code {log['git_sha']} | "
    f"config {log['config_sha256'][:12]} | data {log['dataset_sha256'][:12]} | "
    f"{log['steps']} steps | final loss {final_loss:.4f}"
)